### Mitigating small dataset limitations in disaster image classification via data augmentation

### A case study on the AIDER dataset

Summary

The AIDER dataset (6,433 aerial images across disaster categories) is too small on its own to train a robust classifier. This work builds a geometric and photometric augmentation pipeline - rotation, translation, cropping, zooming, flipping, illumination and color jitter, blurring, and sharpening - to expand effective training data.

A key issue was identified and fixed: torchvision's RandomRotation() introduces black border artifacts absent from real AIDER images, risking incorrect model associations. A custom rotation transform was implemented that crops to the largest axis-aligned border-free region after rotation, removing the artifact entirely.

Training MobileNetV3 with the resulting pipeline doubled the effective training set (5,789 -> 11,578 images) and produced faster convergence and improved accuracy in early training rounds, with both augmented and unmodified datasets reaching similar final accuracy.

##### page break

This work addresses the limited size of the AIDER dataset (6,433 images) by applying augmentation.

Rotation, scale, crop, and flipping are among the most widely used geometric transformations for expanding image datasets, offering a simple and effective way to increase dataset size. These techniques are discussed in:
* Saorj Kumar, Prince Asiamah, Oluwatoyin Jolaoso, Ugochukwu Esiowu - "Enhancing Image Classification with Augmentation: Data Augmentation Techniques for Improved Image Classification" 2025
* Connor Shorten, T. Khoshgoftaar - "A survey on Image Data Augmentation for Deep Learning" 2019
* A. Buslaev, Alex Parinov, Eugene Khvedchenya, V. Iglovikov, Alexandr A Kalinin - "Albumentations: fast and flexible image augmentations" 2018

As recommended by the authors of AIDER:
```
It is advised to further enhance the dataset that random augmentations are probabilistically applied to each image prior to adding it to the batch for training. Specifically there are a number of possible transformations such as geometric (rotations, translations, horizontal axis mirroring, cropping and zooming), as well as image manipulations (illumination changes, color shifting, blurring, sharpening, and shadowing).
```

In [ ]:
import PIL
from matplotlib import pyplot
pyplot.imshow(PIL.Image.open("flood_image0001.jpg"))
pyplot.axis(False)
pyplot.show()

The first image of the "flood" category of the AIDER dataset.

##### page break

In [ ]:
def pyplot_transform(transform):
    image = PIL.Image.open("flood_image0001.jpg")
    fig, axes = pyplot.subplots(1, 5, figsize=(16, 4))
    for i in range(5):
        axes[i].imshow(transform(image))
        axes[i].axis(False)
    pyplot.show()

### rotations
Torchvision RandomRotation() rotates the image by a random angle, but leaves black borders on the sides with no built-in option to fix this.

In [ ]:
from torchvision import transforms
pyplot_transform(transforms.RandomRotation(15))

### translations and cropping
RandomCrop() applies random translation when cropping

In [ ]:
pyplot_transform(transforms.RandomCrop(224))

### horizontal axis mirroring
Horizontal flipping is a simple way to double the dataset size by adding a mirrored version of each image.

In [ ]:
pyplot_transform(transforms.RandomHorizontalFlip())

##### page break

### translations, cropping and zooming
RandomResizedCrop() combines zooming, cropping to a target size, and random translation, and by default also mildly distorts the aspect ratio (0–30%).

In [ ]:
pyplot_transform(transforms.RandomResizedCrop(224, scale=(0.1, 1), ratio=(1, 1)))

### illumination changes
ColorJitter(brightness=0.5, contrast=0.5) randomly adjusts brightness and contrast to simulate different lighting conditions.

In [ ]:
pyplot_transform(transforms.ColorJitter(brightness=0.5, contrast=0.5))

### color shifting
ColorJitter(saturation=0.5, hue=0.5) randomly shifts saturation and hue.

In [ ]:
pyplot_transform(transforms.ColorJitter(saturation=0.5, hue=0.5))

### blurring
Blurring softens fine detail, simulating out-of-focus or lower-resolution capture.

In [ ]:
pyplot_transform(transforms.GaussianBlur(25, sigma=(1, 9)))

##### page break

### sharpening
Sharpening enhances edges and fine detail.

In [ ]:
pyplot_transform(transforms.RandomAdjustSharpness(50))

### all together
Combining the transformations above into a single augmentation pipeline.

In [ ]:
transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0),
    transforms.RandomResizedCrop(224, scale=(0.8, 1), ratio=(1, 1)),
    transforms.GaussianBlur(5, sigma=(0.1, 2.0)),
])
pyplot_transform(transform)

### RandomRotation concerns
RandomRotation() introduces black border artifacts that don't normally occur in the dataset. A model could learn to associate black corners with specific classes rather than genuine image content.

This problem is raised and addressed in:

* Sandhi Wangiyana, P. Samczyński, A. Gromek - "Data Augmentation for Building Footprint Segmentation in SAR Images: An Empirical Study" 2022
* Khaled Alomar, Halil Ibrahim Aysel, Xiaohao Cai - "Data Augmentation in Classification and Segmentation: A Survey and New Strategies" 2023

Alomar et al. propose random local rotation (RLR), which rotates only an internal circular region to avoid boundary artifacts entirely.

An alternative is to crop the image down to the largest border-free area after rotation, which is how we rewrite RandomRotation() below.

##### page break

In [ ]:
import math
def rotatedRectWithMaxArea(w, h, angle):
  """
  https://stackoverflow.com/questions/16702966/rotate-image-and-crop-out-black-borders
  Given a rectangle of size wxh that has been rotated by 'angle' (in
  radians), computes the width and height of the largest possible
  axis-aligned rectangle (maximal area) within the rotated rectangle.
  """
  if w <= 0 or h <= 0:
    return 0,0

  width_is_longer = w >= h
  side_long, side_short = (w,h) if width_is_longer else (h,w)

  # since the solutions for angle, -angle and 180-angle are all the same,
  # if suffices to look at the first quadrant and the absolute values of sin,cos:
  sin_a, cos_a = abs(math.sin(angle)), abs(math.cos(angle))
  if side_short <= 2.*sin_a*cos_a*side_long or abs(sin_a-cos_a) < 1e-10:
    # half constrained case: two crop corners touch the longer side,
    #   the other two corners are on the mid-line parallel to the longer line
    x = 0.5*side_short
    wr,hr = (x/sin_a,x/cos_a) if width_is_longer else (x/cos_a,x/sin_a)
  else:
    # fully constrained case: crop touches all 4 sides
    cos_2a = cos_a*cos_a - sin_a*sin_a
    wr,hr = (w*cos_a - h*sin_a)/cos_2a, (h*cos_a - w*sin_a)/cos_2a

  return wr,hr

import random
class RandomRotation:
    def __init__(self, degrees):
        self.degrees = degrees
    def __call__(self, img):
        angle_deg = random.uniform(-self.degrees, self.degrees)
        return transforms.functional.center_crop(
            transforms.functional.rotate(img, angle_deg, expand=False),
            map(int, rotatedRectWithMaxArea(img.size[1], img.size[0], math.radians(angle_deg))))

pyplot_transform(RandomRotation(30))

##### page break

### all together with improved rotation
The full augmentation pipeline, now using the border-free rotation.

In [ ]:
aider_transform_train = transforms.Compose([
    RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0),
    transforms.RandomResizedCrop(224, scale=(0.8, 1), ratio=(1, 1)),
    transforms.GaussianBlur(5, sigma=(0.1, 2.0)),
])
pyplot_transform(aider_transform_train)

In [ ]:
import pandas
def flwr_accuracy(file, start=0, key="accuracy"):
       df = pandas.read_csv(file + ".csv")
       pyplot.figure(figsize=(8,4))
       for id_val, g in df.groupby("id", sort=False):
              g = g.sort_values("round").tail(len(g) - start)
              pyplot.plot(g["round"], g[key], label=str(id_val))

       pyplot.xlabel("round")
       pyplot.ylabel(key)
       pyplot.grid(True, alpha=0.3)
       pyplot.legend()
       pyplot.tight_layout()
       pyplot.show()

##### page break

### MobileNetV3 on augmented AIDER dataset
noise-multiplier: 0, learning-rate: 0.1, batch-size: 32, frozen backbone

The AIDER dataset (6,433 images) was split into 5,789 training images and 644 test images. Augmentation doubled the training set to 11,578 images by adding one transformed copy for every original image.

The augmented dataset shows improved accuracy in early training rounds, converging to similar accuracy later on.

In [ ]:
flwr_accuracy("aider_augmented", 2)

In [ ]:
flwr_accuracy("aider_augmented", 2, "loss")